# Модуль 5. Разбор кейсов: Поиск аналогов и Танимото

**Модуль 5. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** molecules.csv; битовые множества {1,2,3}/{2,3,4} — искусственное упражнение.

Сравнить ручной бинарный расчёт с молекулярным Morgan и зафиксировать параметры.

**Workflow:** Ручная метрика → Morgan → матрица → ранжирование → ограничение.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab05_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%205_Fingerprinter_Farmacophore.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab05/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. Выбор представления до сравнения

Для бинарных множеств T=|A∩B|/|A∪B|. Для молекул сначала задаём Morgan radius, размер вектора и учёт хиральности. Меняя эти параметры, меняем вопрос, на который отвечает сходство. До вычисления исключаем невалидные графы с сохранением исходных ID в журнале.

**Перед запуском:** Вычислите вручную общие и уникальные биты.

In [2]:
from chemistry import standardize, similarity
from metrics import tanimoto
raw=pd.read_csv(DATA/"molecules.csv",keep_default_na=False)
valid=[]
for row in raw.itertuples():
    try: valid.append((row.id,standardize(row.smiles).parent_smiles))
    except ValueError: pass


[13:59:59] Initializing MetalDisconnector
[13:59:59] Running MetalDisconnector
[13:59:59] Initializing Normalizer
[13:59:59] Running Normalizer
[13:59:59] Initializing MetalDisconnector
[13:59:59] Running MetalDisconnector
[13:59:59] Initializing Normalizer
[13:59:59] Running Normalizer
[13:59:59] Running LargestFragmentChooser
[13:59:59] Running Uncharger
[13:59:59] Initializing MetalDisconnector
[13:59:59] Running MetalDisconnector
[13:59:59] Initializing Normalizer
[13:59:59] Running Normalizer
[13:59:59] Initializing MetalDisconnector
[13:59:59] Running MetalDisconnector
[13:59:59] Initializing Normalizer
[13:59:59] Running Normalizer
[13:59:59] Running LargestFragmentChooser
[13:59:59] Running Uncharger
[13:59:59] Initializing MetalDisconnector
[13:59:59] Running MetalDisconnector
[13:59:59] Initializing Normalizer
[13:59:59] Running Normalizer
[13:59:59] Initializing MetalDisconnector
[13:59:59] Running MetalDisconnector
[13:59:59] Initializing Normalizer
[13:59:59] Running Norma

**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Выбор представления до сравнения — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Матрица и инварианты

Симметрия, единичная диагональ и диапазон [0,1] — необходимые проверки реализации. Они не доказывают биологическую полезность метрики. Хешированные отпечатки допускают коллизии, а выключенный учёт хиральности может объединить различающиеся структуры. Ручная битовая задача служит независимым арифметическим контролем.

**Перед запуском:** Сформируйте matrix всех валидных parent с одинаковым Morgan-параметрами.

In [3]:
matrix=np.array([[similarity(a[1],b[1]) for b in valid] for a in valid])
assert tanimoto({1,2,3},{2,3,4})==0.5
assert np.allclose(matrix,matrix.T) and np.allclose(np.diag(matrix),1)
assert matrix.min()>=0 and matrix.max()<=1


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Матрица и инварианты — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Ранжирование аналогов

Сохраняем сходство с запросом вместе с ID и параметрами. Верхние строки могут быть альтернативными сериализациями одной структуры. Для разнообразной закупки нужен дополнительный критерий выбора, например группировка по каркасам. Порог T нельзя переносить между представлениями без отдельной проверки.

**Перед запуском:** Отранжируйте аналоги этанола; сохраните ID и параметры, не приписывая активность.

In [4]:
ranking=pd.DataFrame({"id":[x[0] for x in valid],"T_to_ethanol":[similarity("CCO",x[1]) for x in valid]})
display(ranking.sort_values("T_to_ethanol",ascending=False))
pd.DataFrame(matrix,index=[x[0] for x in valid],columns=[x[0] for x in valid]).to_csv(OUT/"tanimoto.csv")
plt.imshow(matrix,vmin=0,vmax=1);plt.colorbar(label="Morgan Tanimoto")
plt.savefig(OUT/"similarity.png",dpi=150);plt.show()
artifact={"radius":2,"fpSize":2048,"includeChirality":True,"molecules":len(valid)}


,id,T_to_ethanol
0,M01,1.000000
1,M02,1.000000
9,M12,0.375000
2,M03,0.181818
3,M04,0.181818
7,M10,0.133333
8,M11,0.133333
5,M06,0.066667
4,M05,0.000000
6,M07,0.000000


/tmp/ipykernel_14247/288592287.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(OUT/"similarity.png",dpi=150);plt.show()


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Ранжирование аналогов — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

T=0,5 на множествах; диагональ Morgan=1; симметрия и диапазон [0,1].

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':5,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab05/worked/benchmark.json


## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab05_practice.ipynb](lab05_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.